<a href="https://colab.research.google.com/github/anhthu512/fall-detection/blob/main/fall_dectect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install roboflow


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="")

In [ ]:
from roboflow import Roboflow
import os

API_KEY = "4g9YjwFQeDt4rtz69iLp"  # ← thay bằng key của bạn
rf = Roboflow(api_key=API_KEY)

datasets = [
    # (workspace,                              project,                                    version)
    ("testspace-u8f8z",                        "humanpose",                                1),
    ("roboflow-universe-projects",             "fall-detection-ca3o8",                     1),
    ("fordataset",                             "yolov7-hiqkc",                             1),
    ("roboflow-universe-projects",             "personal-protective-equipment-combined-model", 1),
    ("murderer-666-9rxdt",                     "fall3",                                    1),
    ("roboflow-ngkro",                         "cleaneddataset",                           1),
]

download_paths = {}
for ws, proj, ver in datasets:
    print(f"\n📥 Downloading {proj}...")
    project  = rf.workspace(ws).project(proj)
    dataset  = project.version(ver).download("yolov8", location=f"/kaggle/working/{proj}")
    download_paths[proj] = f"/kaggle/working/{proj}"
    print(f"✅ Saved to /kaggle/working/{proj}")


📥 Downloading humanpose...
loading Roboflow workspace...
loading Roboflow project...
✅ Saved to /kaggle/working/humanpose

📥 Downloading fall-detection-ca3o8...
loading Roboflow workspace...
loading Roboflow project...
✅ Saved to /kaggle/working/fall-detection-ca3o8

📥 Downloading yolov7-hiqkc...
loading Roboflow workspace...
loading Roboflow project...
✅ Saved to /kaggle/working/yolov7-hiqkc

📥 Downloading personal-protective-equipment-combined-model...
loading Roboflow workspace...
loading Roboflow project...
✅ Saved to /kaggle/working/personal-protective-equipment-combined-model

📥 Downloading fall3...
loading Roboflow workspace...
loading Roboflow project...
✅ Saved to /kaggle/working/fall3

📥 Downloading cleaneddataset...
loading Roboflow workspace...
loading Roboflow project...
✅ Saved to /kaggle/working/cleaneddataset


In [ ]:
# CELL 3 – Xem classes của từng dataset
import yaml
import os

for proj, path in download_paths.items():
    yaml_path = os.path.join(path, "data.yaml")
    if os.path.exists(yaml_path):
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        print(f"\n📋 {proj}: nc={cfg['nc']}, names={cfg['names']}")
    else:
        print(f"\n⚠️ {proj}: Không tìm thấy data.yaml tại {yaml_path}")


📋 humanpose: nc=4, names=['Fall', 'Sitting', 'Standing', 'safety_equipment']

📋 fall-detection-ca3o8: nc=1, names=['Fall-Detected']

📋 yolov7-hiqkc: nc=1, names=['fall']

📋 personal-protective-equipment-combined-model: nc=14, names=['Fall-Detected', 'Gloves', 'Goggles', 'Hardhat', 'Ladder', 'Mask', 'NO-Gloves', 'NO-Goggles', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest']

📋 fall3: nc=1, names=['fall']

📋 cleaneddataset: nc=14, names=['Fall-Detected', 'Gloves', 'Goggles', 'Hardhat', 'Ladder', 'Mask', 'NO-Gloves', 'NO-Goggles', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest']


In [ ]:
# ============================================================
# CELL 4 – MAPPINGS chính xác (0=fall, 1=standing, None=bỏ)
# ============================================================
import yaml, os, shutil

MAPPINGS = {
    "humanpose": {
        0: 0,     # Fall        → fall
        1: None,  # Sitting     → bỏ
        2: 1,     # Standing    → standing
        3: None,  # safety_equip→ bỏ
    },
    "fall-detection-ca3o8": {
        0: 0,     # Fall-Detected → fall
        # không có standing trong dataset này
    },
    "yolov7-hiqkc": {
        0: 0,     # fall → fall
    },
    "personal-protective-equipment-combined-model": {
        0:  0,    # Fall-Detected   → fall
        1:  None, # Gloves          → bỏ
        2:  None, # Goggles         → bỏ
        3:  None, # Hardhat         → bỏ
        4:  None, # Ladder          → bỏ
        5:  None, # Mask            → bỏ
        6:  None, # NO-Gloves       → bỏ
        7:  None, # NO-Goggles      → bỏ
        8:  None, # NO-Hardhat      → bỏ
        9:  None, # NO-Mask         → bỏ
        10: None, # NO-Safety Vest  → bỏ
        11: 1,    # Person          → standing
        12: None, # Safety Cone     → bỏ
        13: None, # Safety Vest     → bỏ
    },
    "fall3": {
        0: 0,     # fall → fall
    },
    "cleaneddataset": {
        0:  0,    # Fall-Detected   → fall
        1:  None,
        2:  None,
        3:  None,
        4:  None,
        5:  None,
        6:  None,
        7:  None,
        8:  None,
        9:  None,
        10: None,
        11: 1,    # Person          → standing
        12: None,
        13: None,
    },
}

# ============================================================
# CELL 5 (FIX) – Dùng prefix ngắn để tránh lỗi tên file dài
# ============================================================
import yaml, os, shutil

MERGED = "/kaggle/working/merged"
for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED}/images/{split}", exist_ok=True)
    os.makedirs(f"{MERGED}/labels/{split}", exist_ok=True)

# Prefix ngắn cho từng dataset
SHORT_PREFIX = {
    "humanpose":                                    "hp",
    "fall-detection-ca3o8":                         "fd",
    "yolov7-hiqkc":                                 "y7",
    "personal-protective-equipment-combined-model": "ppe",  # ← từ đây fix lỗi
    "fall3":                                        "f3",
    "cleaneddataset":                               "cd",
}

def remap_labels(label_dir, mapping_dict, output_dir, prefix):
    os.makedirs(output_dir, exist_ok=True)
    total, skipped = 0, 0
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, fname)) as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_class = int(parts[0])
            new_class = mapping_dict.get(old_class)
            if new_class is None:
                skipped += 1
                continue
            parts[0] = str(new_class)
            new_lines.append(" ".join(parts))
            total += 1
        # Tên file output dùng prefix ngắn
        out_fname = f"{prefix}_{fname}"
        with open(os.path.join(output_dir, out_fname), "w") as f:
            f.write("\n".join(new_lines))
    return total, skipped

print("🔄 Bắt đầu remap + merge...\n")
for proj, base_path in download_paths.items():
    mapping = MAPPINGS[proj]
    prefix  = SHORT_PREFIX[proj]
    proj_total, proj_skip = 0, 0

    for split in ["train", "valid", "test"]:
        img_src = os.path.join(base_path, split, "images")
        lbl_src = os.path.join(base_path, split, "labels")
        if not os.path.exists(img_src):
            continue

        # Copy ảnh với prefix ngắn + truncate tên file nếu vẫn dài
        for img_file in os.listdir(img_src):
            ext      = os.path.splitext(img_file)[1]          # .jpg
            stem     = os.path.splitext(img_file)[0][:60]     # giới hạn 60 ký tự
            new_name = f"{prefix}_{stem}{ext}"
            shutil.copy2(
                os.path.join(img_src, img_file),
                os.path.join(MERGED, "images", split, new_name)
            )

        # Remap labels (cùng cách đặt tên để ảnh khớp label)
        tmp_dir = f"/tmp/labels_{proj}_{split}"
        # Xóa tmp cũ nếu có để tránh duplicate
        if os.path.exists(tmp_dir):
            shutil.rmtree(tmp_dir)
        t, s = remap_labels(lbl_src, mapping, tmp_dir, prefix)

        for lbl_file in os.listdir(tmp_dir):
            shutil.copy2(
                os.path.join(tmp_dir, lbl_file),
                os.path.join(MERGED, "labels", split, lbl_file)
            )
        proj_total += t; proj_skip += s

    print(f"✅ {proj}: {proj_total} kept, {proj_skip} skipped")

# ============================================================
# CELL 6 – data.yaml + thống kê
# ============================================================
merged_yaml = {
    "path": MERGED,
    "train": "images/train",
    "val":   "images/valid",
    "test":  "images/test",
    "nc":    2,
    "names": ["fall", "standing"]
}
with open(f"{MERGED}/data.yaml", "w") as f:
    yaml.dump(merged_yaml, f, default_flow_style=False, allow_unicode=True)

for split in ["train", "valid", "test"]:
    n_img = len(os.listdir(f"{MERGED}/images/{split}"))
    n_lbl = len(os.listdir(f"{MERGED}/labels/{split}"))
    print(f"📊 {split}: {n_img} images, {n_lbl} labels")

print("\n✅ Hoàn tất! data.yaml:")
print(yaml.dump(merged_yaml))



🔄 Bắt đầu remap + merge...

✅ humanpose: 12690 kept, 12306 skipped
✅ fall-detection-ca3o8: 4498 kept, 0 skipped
✅ yolov7-hiqkc: 601 kept, 0 skipped
✅ personal-protective-equipment-combined-model: 5947 kept, 103619 skipped
✅ fall3: 4585 kept, 0 skipped
✅ cleaneddataset: 5756 kept, 75582 skipped
📊 train: 60592 images, 61116 labels
📊 valid: 16802 images, 16915 labels
📊 test: 8975 images, 9034 labels

✅ Hoàn tất! data.yaml:
names:
- fall
- standing
nc: 2
path: /kaggle/working/merged
test: images/test
train: images/train
val: images/valid



In [ ]:
# ============================================================
# CELL 7 – Kiểm tra class imbalance trước khi train
# ============================================================
import os
from collections import Counter

def count_classes(label_dir):
    counter = Counter()
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    return counter

train_counts = count_classes(f"{MERGED}/labels/train")
total = sum(train_counts.values())
print("📊 Class distribution (train):")
for cls_id, name in enumerate(["fall", "standing"]):
    cnt = train_counts.get(cls_id, 0)
    print(f"  Class {cls_id} ({name}): {cnt:,} ({cnt/total*100:.1f}%)")

ratio = train_counts.get(0, 1) / max(train_counts.get(1, 1), 1)
print(f"\n⚖️  fall/standing ratio = {ratio:.2f}x")
if ratio > 3:
    print("⚠️  Mất cân bằng cao – nên dùng cls weight")


📊 Class distribution (train):
  Class 0 (fall): 15,707 (63.0%)
  Class 1 (standing): 9,221 (37.0%)

⚖️  fall/standing ratio = 1.70x


In [ ]:
import subprocess, sys

# ⚠️ Trên Kaggle Notebook, gọi model.train(device=[0,1]) trực tiếp
# thường bị HANG. Cách đúng là chạy qua subprocess (DDP safe).

train_script = """
from ultralytics import YOLO

MERGED = "/kaggle/working/merged"

model = YOLO("yolov8m.pt")

model.train(
    data=f"{MERGED}/data.yaml",

    # ── Cơ bản ───────────────────────────────────────────────
    epochs        = 350,
    imgsz         = 640,
    batch         = 32,        # ← 2 GPU: 16/GPU × 2 = 32 total (linear scaling)
    workers       = 8,         # ← 4 workers/GPU × 2 GPU
    device        = [0, 1],    # ← QUAN TRỌNG: bật DDP multi-GPU
    project       = "/kaggle/working/runs",
    name          = "fall_v1",
    exist_ok      = True,

    # ── Compiler (PyTorch 2.x) ────────────────────────────────
    compile       = True,      # ← torch.compile tăng ~10-20% tốc độ

    # ── Cache ảnh vào RAM ─────────────────────────────────────
    # 98K ảnh × ~27KB ≈ 2.7GB RAM — Kaggle có ~30GB RAM, hoàn toàn ổn
    cache         = "ram",     # ← loại bỏ disk I/O bottleneck

    # ── Optimizer ────────────────────────────────────────────
    optimizer     = "AdamW",
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,

    # ── Warmup ───────────────────────────────────────────────
    warmup_epochs   = 5,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,

    # ── Early Stopping ───────────────────────────────────────
    patience      = 20,

    # ── Regularization ───────────────────────────────────────
    dropout       = 0.1,
    # label_smoothing đã bị deprecated, bỏ hoàn toàn

    # ── Loss weights ─────────────────────────────────────────
    cls           = 0.5,
    box           = 7.5,
    dfl           = 1.5,

    # ── Augmentation ─────────────────────────────────────────
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 10,
    translate     = 0.1,
    scale         = 0.5,
    shear         = 2.0,
    flipud        = 0.3,
    fliplr        = 0.5,
    mosaic        = 1.0,
    mixup         = 0.1,
    copy_paste    = 0.1,
    erasing       = 0.3,

    # ── AMP (đã bật mặc định) ─────────────────────────────────
    amp           = True,

    # ── Eval & Save ──────────────────────────────────────────
    val           = True,
    save          = True,
    save_period   = 10,
    plots         = True,
    verbose       = True,
)
"""

# Ghi script ra file rồi gọi qua subprocess để tránh DDP hang trong notebook
with open("/kaggle/working/train_ddp.py", "w") as f:
    f.write(train_script)

try:
    result = subprocess.run(
        [sys.executable, "/kaggle/working/train_ddp.py"],
        check=True,
        capture_output=True, # Capture stdout and stderr
        text=True # Decode stdout and stderr as text
    )
    print("Subprocess stdout:", result.stdout)
except subprocess.CalledProcessError as e:
    print("Subprocess failed with exit code:", e.returncode)
    print("Subprocess stdout:", e.stdout)
    print("Subprocess stderr:", e.stderr)
    raise # Re-raise the exception after printing details

CalledProcessError: Command '['/usr/bin/python3', '/kaggle/working/train_ddp.py']' returned non-zero exit status 1.

In [ ]:
# ============================================================
# CELL 10 – INFERENCE / TEST MODEL FALL DETECTION
# Model đã train: class 0 = fall, class 1 = standing
# ============================================================

import os
import zipfile
import glob
from pathlib import Path
from collections import Counter

import torch
from IPython.display import Image, display

# Nếu Kaggle chưa có ultralytics thì tự cài
try:
    from ultralytics import YOLO
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])
    from ultralytics import YOLO


# ============================================================
# 1. CHỌN WEIGHTS
# ============================================================
# Path mặc định sau cell train của notebook này
WEIGHTS = Path("/kaggle/working/runs/fall_v1/weights/best.pt")

# Nếu path mặc định không tồn tại, tự tìm best.pt mới nhất trong /kaggle/working/runs
if not WEIGHTS.exists():
    candidates = sorted(
        Path("/kaggle/working/runs").glob("**/weights/best.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Không tìm thấy best.pt. Hãy train xong model trước, "
            "hoặc sửa biến WEIGHTS trỏ đúng tới file best.pt."
        )
    WEIGHTS = candidates[0]

print(f"✅ Using weights: {WEIGHTS}")


# ============================================================
# 2. CHỌN SOURCE ĐỂ INFERENCE
# ============================================================
# Cách dùng:
# - Muốn test folder ảnh: SOURCE = "/kaggle/working/merged/images/test"
# - Muốn test 1 ảnh:     SOURCE = "/kaggle/input/xxx/image.jpg"
# - Muốn test 1 video:   SOURCE = "/kaggle/input/xxx/video.mp4"
# - Để SOURCE = "" thì cell sẽ tự tìm test folder hoặc tự giải nén merged_dataset.zip nếu có.

SOURCE = ""   # ← sửa dòng này nếu muốn inference ảnh/video/folder riêng


def auto_find_source():
    """
    Tự tìm dữ liệu test.
    Ưu tiên:
    1) /kaggle/working/merged/images/test
    2) /kaggle/working/merged/images/valid
    3) Nếu merged đã bị xóa nhưng còn merged_dataset.zip thì giải nén ra /kaggle/working/infer_dataset
    """
    direct_candidates = [
        Path("/kaggle/working/merged/images/test"),
        Path("/kaggle/working/merged/images/valid"),
    ]

    for p in direct_candidates:
        if p.exists():
            return str(p)

    # Nếu đã chạy cell zip và xóa folder merged, vẫn có thể còn zip
    zip_candidates = [Path("/kaggle/working/merged_dataset.zip")]
    zip_candidates += list(Path("/kaggle/input").glob("**/merged_dataset.zip"))

    zip_candidates = [p for p in zip_candidates if p.exists()]
    if len(zip_candidates) > 0:
        zip_path = zip_candidates[0]
        extract_dir = Path("/kaggle/working/infer_dataset")
        test_dir = extract_dir / "merged" / "images" / "test"
        valid_dir = extract_dir / "merged" / "images" / "valid"

        if not test_dir.exists() and not valid_dir.exists():
            print(f"📦 Extracting dataset zip: {zip_path}")
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)

        if test_dir.exists():
            return str(test_dir)
        if valid_dir.exists():
            return str(valid_dir)

    raise FileNotFoundError(
        "Không tự tìm thấy source để inference.\n"
        "Hãy sửa biến SOURCE thành path ảnh/video/folder, ví dụ:\n"
        'SOURCE = "/kaggle/input/ten-dataset/video.mp4"\n'
        'SOURCE = "/kaggle/input/ten-dataset/image.jpg"\n'
        'SOURCE = "/kaggle/working/merged/images/test"'
    )


if SOURCE.strip() == "":
    SOURCE = auto_find_source()

SOURCE = str(SOURCE)
print(f"✅ Inference source: {SOURCE}")


# ============================================================
# 3. LOAD MODEL
# ============================================================
device = 0 if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")

model = YOLO(str(WEIGHTS))

# Đảm bảo tên class hiển thị đúng
model.names = {
    0: "fall",
    1: "standing"
}


# ============================================================
# 4. CHẠY INFERENCE
# ============================================================
OUT_PROJECT = "/kaggle/working/runs"
OUT_NAME    = "fall_v1_inference"

CONF = 0.25     # confidence threshold: thấp hơn thì bắt nhiều hơn nhưng dễ false positive
IOU  = 0.50     # NMS IoU threshold
IMGSZ = 640     # phải khớp với imgsz train trong notebook này

results_generator = model.predict(
    source      = SOURCE,
    imgsz       = IMGSZ,
    conf        = CONF,
    iou         = IOU,
    device      = device,
    save        = True,     # lưu ảnh/video đã vẽ bbox
    save_txt    = True,     # lưu label dự đoán dạng YOLO txt
    save_conf   = True,     # lưu kèm confidence trong txt
    project     = OUT_PROJECT,
    name        = OUT_NAME,
    exist_ok    = True,
    stream      = True,     # tiết kiệm RAM khi source là folder/video lớn
    verbose     = False
)


# ============================================================
# 5. THỐNG KÊ KẾT QUẢ
# ============================================================
class_counter = Counter()
num_frames_or_images = 0
num_boxes = 0

for r in results_generator:
    num_frames_or_images += 1

    if r.boxes is None or len(r.boxes) == 0:
        continue

    cls_ids = r.boxes.cls.detach().cpu().numpy().astype(int).tolist()
    class_counter.update(cls_ids)
    num_boxes += len(cls_ids)

print("\n================ INFERENCE SUMMARY ================")
print(f"Source processed : {num_frames_or_images}")
print(f"Total boxes      : {num_boxes}")
print(f"fall boxes       : {class_counter.get(0, 0)}")
print(f"standing boxes   : {class_counter.get(1, 0)}")

if class_counter.get(0, 0) > 0:
    print("⚠️ Có phát hiện FALL trong source.")
else:
    print("✅ Không thấy FALL theo ngưỡng CONF hiện tại.")

OUT_DIR = Path(OUT_PROJECT) / OUT_NAME
print(f"\n✅ Output saved to: {OUT_DIR}")


# ============================================================
# 6. HIỂN THỊ MỘT VÀI ẢNH KẾT QUẢ TRONG NOTEBOOK
# ============================================================
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
video_exts = {".mp4", ".avi", ".mov", ".mkv"}

output_files = sorted([p for p in OUT_DIR.glob("**/*") if p.is_file()])
output_images = [p for p in output_files if p.suffix.lower() in image_exts]
output_videos = [p for p in output_files if p.suffix.lower() in video_exts]

if len(output_images) > 0:
    print(f"\n🖼️ Hiển thị tối đa 6 ảnh kết quả đầu tiên:")
    for p in output_images[:6]:
        print(p)
        display(Image(filename=str(p), width=800))

elif len(output_videos) > 0:
    print("\n🎬 Source là video. Video output nằm ở:")
    for p in output_videos:
        print(p)
    print("\nNếu muốn xem trực tiếp trên Kaggle, mở file output ở panel bên phải hoặc tải về.")

else:
    print("\n⚠️ Không tìm thấy file ảnh/video output để preview, nhưng label txt có thể đã được lưu trong thư mục labels.")
